# NBF: Canonical manuscript figures (Phase F1/F2)

All figures are rendered from persisted CSVs and joblib models written by
Phases 2-11. Every feature construction and model load goes through the
Phase 3R winning configuration JSON via src.plot_style.load_winning_config.
No feature-set name is hard-coded.

Figure roster:
  1  Fig 01  P-T distribution
  2  Fig 02  PCA biplot
  3  Fig 03  Pred vs obs, pressure
  4  Fig 04  Pred vs obs, temperature
  5  Fig 05  Model comparison bar plot
  6  Fig 06  LOSO pooled RMSE
  7  Fig 07  SHAP beeswarm P
  8  Fig 08  SHAP beeswarm T
  9  Fig 09  SHAP dependence panels
 10  Fig 10  Putirka vs ML
 11  Fig 11  Bias-correction residuals (4 tracks)
 12  Fig 12  ArcPL natural samples on geotherm
 13  Fig 13  IsolationForest OOD score vs residual
 14  Fig 14  MC analytical uncertainty vs IQR uncertainty

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Canonical features, models, and plotting from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr
from src.geotherm import hasterok_chapman_geotherm
from src.plot_style import (
    apply_style, panel_label, stats_box, one_to_one, regression_line,
    save_figure, fmt_value,
    load_winning_config, canonical_model_filename,
    TOL_BRIGHT, MODEL_COLORS, PUTIRKA_COLOR, ML_COLOR,
)

apply_style()


In [ ]:
# Load Phase 3R winning config once. All downstream feature-matrix construction
# and canonical model filenames route through WIN_FEAT.
# v7 Part H: per-family canonical spec replaces the legacy
# legacy pre-v7 single-winner assumption. Forest / T_C / opx_liq is the
# primary single-feature-set stand-in for notebooks that have not
# been restructured into per-family output blocks.
from src.data import canonical_model_spec
_spec_primary = canonical_model_spec('T_C', 'opx_liq', 'forest', RESULTS)
WIN_FEAT = _spec_primary['feature_set']
print(f'v7 primary (forest / T_C / opx_liq) feature set: {WIN_FEAT}')
feat_fn = lambda df, use_liq: build_feature_matrix(df, WIN_FEAT, use_liq=use_liq)
print(f'Phase 3R winning feature set: {WIN_FEAT}')


In [ ]:
# File audit: check that every input NBF reads is present before plotting.
# Missing files surface as a single table rather than as a traceback inside a
# figure cell partway through the notebook.
_required = [
    # (path, owner_notebook, used_by_fig)
    (DATA_PROC / 'opx_clean_opx_liq.parquet',       'nb01', 'fig01,fig02,fig14'),
    (DATA_PROC / 'opx_clean_opx_only.parquet',      'nb01', 'fig03,fig04'),
    (DATA_SPLITS / 'test_indices_opx_liq.npy',      'nb01', 'fig03,fig04,fig14'),
    (DATA_SPLITS / 'test_indices_opx.npy',          'nb01', 'fig03,fig04'),
    (RESULTS / 'nb03_per_family_winners.json',          'nb03', 'all'),
    (RESULTS / 'nb03_multi_seed_results.csv',       'nb03', 'fig05'),
    (RESULTS / 'nb04_putirka_vs_ml.csv',            'nb04', 'fig10 (legacy)'),
    (RESULTS / 'nb04b_arcpl_predictions.csv',       'nb04b', 'fig13'),
    (RESULTS / 'nb05_loso_pooled.csv',              'nb05', 'fig06'),
    (RESULTS / 'nb07_test_predictions.csv',         'nb07', 'fig11'),
    (RESULTS / 'nb10_arcpl_ood_scores.csv',         'nb10', 'fig12,fig13'),
    (RESULTS / 'nb10_mc_per_sample.csv',            'nb10', 'fig14'),
]

_rows = []
for _p, _owner, _fig in _required:
    _ok = _p.exists()
    _size = _p.stat().st_size if _ok else 0
    _rows.append({
        'path':      _p.relative_to(ROOT).as_posix(),
        'owner':     _owner,
        'used_by':   _fig,
        'exists':    _ok,
        'bytes':     _size,
        'status':    'OK' if _ok and _size > 10 else ('EMPTY' if _ok else 'MISSING'),
    })

_audit = pd.DataFrame(_rows)
_audit.to_csv(RESULTS / 'nbF_file_audit.csv', index=False)
print('NBF input audit:')
print(_audit.to_string(index=False))
_bad = _audit[_audit['status'] != 'OK']
if len(_bad) > 0:
    print(f'\nWARNING: {len(_bad)} input(s) missing or empty. Affected figures will be skipped.')
else:
    print('\nAll inputs present.')


In [ ]:
# Fig 01: P-T distribution of the opx-liq training set
df_liq = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(df_liq['T_C'], df_liq['P_kbar'],
                c=df_liq['Mg_num'], cmap='viridis',
                s=12, alpha=0.7, edgecolor='none')
cb = plt.colorbar(sc, ax=ax, label='opx Mg#')
ax.set_xlabel('T (C)')
ax.set_ylabel('P (kbar)')
ax.invert_yaxis()
ax.set_title(f'opx-liq experimental distribution (n = {len(df_liq)})')
ax.grid(True, alpha=0.3)
save_figure(fig, FIGURES / 'fig01_pt_distribution',
            caption='Figure 1. P-T distribution of the opx-liq experimental dataset, colored by orthopyroxene Mg#.')
plt.close(fig)

In [ ]:
# Fig 02: PCA biplot over opx oxides (colored by T)
oxide_cols = [c for c in ['SiO2', 'TiO2', 'Al2O3', 'Cr2O3', 'FeO_total', 'MnO',
                          'MgO', 'CaO', 'Na2O'] if c in df_liq.columns]
X_pca = StandardScaler().fit_transform(df_liq[oxide_cols].fillna(0.0))
pca = PCA(n_components=2).fit(X_pca)
Xp = pca.transform(X_pca)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(Xp[:, 0], Xp[:, 1], c=df_liq['T_C'], cmap='magma',
                s=10, alpha=0.7, edgecolor='none')
for i, name in enumerate(oxide_cols):
    v = pca.components_[:, i] * 3
    ax.arrow(0, 0, v[0], v[1], color='k', lw=0.8, alpha=0.8,
             head_width=0.08, length_includes_head=True)
    ax.text(v[0] * 1.08, v[1] * 1.08, name, fontsize=8, ha='center', va='center')
plt.colorbar(sc, ax=ax, label='T (C)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('opx PCA biplot')
ax.grid(True, alpha=0.3)
save_figure(fig, FIGURES / 'fig02_pca_biplot',
            caption='Figure 2. Principal-component biplot of orthopyroxene oxide compositions, colored by T.')
plt.close(fig)

In [ ]:
# Fig 03 and Fig 04: pred vs obs P and T (opx_only and opx_liq side by side)
# Uses canonical model filenames for the winning feature set.

idx_te_liq = np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')
idx_te_opx = np.load(DATA_SPLITS / 'test_indices_opx.npy')
df_te_liq = df_liq.loc[idx_te_liq].copy()
df_opx = pd.read_parquet(DATA_PROC / 'opx_clean_opx_only.parquet')
df_te_opx = df_opx.loc[idx_te_opx].copy()

# Per-(track, target) canonical feature matrices. Forest winners differ
# across combos (alr/raw/pwlr), so we cannot reuse a single feat_fn.
from src.data import canonical_model_spec
_combos = [('opx_only', 'P_kbar', False), ('opx_only', 'T_C', False),
           ('opx_liq',  'P_kbar', True),  ('opx_liq',  'T_C',  True)]
X_te = {}
for _tr, _tg, _ul in _combos:
    _spec = canonical_model_spec(_tg, _tr, 'forest', RESULTS)
    _df = df_te_liq if _tr == 'opx_liq' else df_te_opx
    X_te[(_tr, _tg)], _ = build_feature_matrix(_df, _spec['feature_set'], use_liq=_ul)

# Legacy aliases so downstream code that still references X_liq_te / X_opx_te
# keeps working for target-agnostic uses (not used by the pred_vs_obs blocks
# below, which switch to X_te[(...)]).
X_liq_te = X_te[('opx_liq', 'T_C')]
X_opx_te = X_te[('opx_only', 'T_C')]

models = {
    ('opx_only', 'P_kbar'): joblib.load(MODELS / canonical_model_filename('P_kbar', 'opx_only', 'forest', RESULTS)),
    ('opx_only', 'T_C'):    joblib.load(MODELS / canonical_model_filename('T_C', 'opx_only', 'forest', RESULTS)),
    ('opx_liq',  'P_kbar'): joblib.load(MODELS / canonical_model_filename('P_kbar', 'opx_liq', 'forest',  RESULTS)),
    ('opx_liq',  'T_C'):    joblib.load(MODELS / canonical_model_filename('T_C', 'opx_liq', 'forest',  RESULTS)),
}


def pred_vs_obs(ax, y_true, y_pred, unit, color):
    ax.scatter(y_true, y_pred, s=18, alpha=0.6, color=color, edgecolor='k', lw=0.3)
    one_to_one(ax)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    slope, intercept = regression_line(ax, y_true, y_pred, color='gray', lw=0.8, ls='--')
    stats_box(ax, rmse=rmse, r2=r2, slope=slope, intercept=intercept,
              n=len(y_true), unit=unit)
    ax.set_xlabel(f'Observed {unit}')
    ax.set_ylabel(f'Predicted {unit}')
    ax.grid(True, alpha=0.3)


# Figure 3: pressure
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
pred = predict_median(models[('opx_only', 'P_kbar')], X_te[('opx_only', 'P_kbar')])
pred_vs_obs(axes[0], df_te_opx['P_kbar'].values, pred, 'kbar', MODEL_COLORS['RF'])
axes[0].set_title('RF opx-only')
panel_label(axes[0], 'a')
pred = predict_median(models[('opx_liq', 'P_kbar')], X_te[('opx_liq', 'P_kbar')])
pred_vs_obs(axes[1], df_te_liq['P_kbar'].values, pred, 'kbar', MODEL_COLORS['RF'])
axes[1].set_title('RF opx-liq')
panel_label(axes[1], 'b')
plt.tight_layout()
save_figure(fig, FIGURES / 'fig03_pred_vs_obs_P',
            caption='Figure 3. Test-set predictions vs observations for pressure (opx-only and opx-liq).')
plt.close(fig)

# Figure 4: temperature
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
pred = predict_median(models[('opx_only', 'T_C')], X_te[('opx_only', 'T_C')])
pred_vs_obs(axes[0], df_te_opx['T_C'].values, pred, 'C', MODEL_COLORS['RF'])
axes[0].set_title('RF opx-only')
panel_label(axes[0], 'a')
pred = predict_median(models[('opx_liq', 'T_C')], X_te[('opx_liq', 'T_C')])
pred_vs_obs(axes[1], df_te_liq['T_C'].values, pred, 'C', MODEL_COLORS['RF'])
axes[1].set_title('RF opx-liq')
panel_label(axes[1], 'b')
plt.tight_layout()
save_figure(fig, FIGURES / 'fig04_pred_vs_obs_T',
            caption='Figure 4. Test-set predictions vs observations for temperature (opx-only and opx-liq).')
plt.close(fig)

In [ ]:
# Fig 05: model comparison bar plot (mean +/- std across 20 split seeds)
multi = pd.read_csv(RESULTS / 'nb03_multi_seed_results.csv')
multi_win = multi[multi['feature_set'] == WIN_FEAT].copy()

agg = (multi_win
       .groupby(['model_name', 'track', 'target'])['rmse_test']
       .agg(['mean', 'std']).reset_index())

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharey=False)
combos = [('opx_only', 'T_C', 'C'),    ('opx_only', 'P_kbar', 'kbar'),
          ('opx_liq',  'T_C', 'C'),    ('opx_liq',  'P_kbar', 'kbar')]
for ax, (tr, tgt, unit) in zip(axes.flat, combos):
    sub = agg[(agg.track == tr) & (agg.target == tgt)].set_index('model_name')
    order = [m for m in ['RF', 'ERT', 'XGB', 'GB'] if m in sub.index]
    means = sub.loc[order, 'mean'].values
    stds = sub.loc[order, 'std'].values
    colors = [MODEL_COLORS[m] for m in order]
    ax.bar(order, means, yerr=stds, capsize=4, color=colors, edgecolor='k', lw=0.6)
    ax.set_ylabel(f'Test RMSE ({unit})')
    ax.set_title(f'{tr} / {tgt}')
    ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
save_figure(fig, FIGURES / 'fig05_model_comparison',
            caption='Figure 5. Test-set RMSE across four model families on both tracks and targets. Error bars show std over 20 split seeds.')
plt.close(fig)

In [ ]:
# Fig 06: LOSO / cluster-kfold / gridded-PT pooled RMSE for the canonical RF.
# nb05 emits one row per (strategy, model, target); filter to RF + WIN_FEAT so
# the comparison across strategies is apples-to-apples with the manuscript RF.
loso = pd.read_csv(RESULTS / 'nb05_loso_pooled.csv')
loso_rf = loso[(loso['model_name'] == 'RF') & (loso['feature_set'] == WIN_FEAT)].copy()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, tgt, unit in zip(axes, ['T_C', 'P_kbar'], ['C', 'kbar']):
    sub = loso_rf[loso_rf.target == tgt]
    order = sub['strategy'].tolist()
    vals = sub['rmse'].values
    ax.bar(order, vals, color=[TOL_BRIGHT['blue'], TOL_BRIGHT['green'], TOL_BRIGHT['yellow']],
           edgecolor='k', lw=0.6)
    ax.set_ylabel(f'Pooled RMSE ({unit})')
    ax.set_title(f'{tgt}')
    ax.grid(True, alpha=0.3, axis='y')
    for x, v in zip(order, vals):
        ax.text(x, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
save_figure(fig, FIGURES / 'fig06_loso_pooled',
            caption='Figure 6. Pooled out-of-fold RMSE across three group-aware CV strategies (LOSO, cluster-KFold, gridded PT) for the canonical RF on the winning feature set.')
plt.close(fig)


In [ ]:
# Figs 07-09: SHAP figures are rendered directly by nb06 (beeswarms and
# dependence panels). Here we just verify they exist and copy their captions
# into the figures folder as .txt for the manuscript bundle.
shap_figs = [
    ('fig07_shap_P_beeswarm',  'Figure 7. SHAP beeswarm for the pressure model.'),
    ('fig08_shap_T_beeswarm',  'Figure 8. SHAP beeswarm for the temperature model.'),
    ('fig09_shap_P_dependence', 'Figure 9a. SHAP dependence plots for the top three pressure features.'),
    ('fig09_shap_T_dependence', 'Figure 9b. SHAP dependence plots for the top three temperature features.'),
]
for base, cap in shap_figs:
    png = FIGURES / f'{base}.png'
    if png.exists():
        with open(FIGURES / f'{base}.txt', 'w') as f:
            f.write(cap + '\n')
    else:
        print(f'missing SHAP figure: {png.name}')

In [ ]:
# Fig 10: Putirka (Thermobar) vs ML pred-vs-obs
putirka_path = RESULTS / 'nb04_putirka_vs_ml.csv'
if putirka_path.exists():
    pu = pd.read_csv(putirka_path)
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, (src, tgt, unit) in zip(axes.flat, [
        ('putirka', 'T_C', 'C'), ('ml', 'T_C', 'C'),
        ('putirka', 'P_kbar', 'kbar'), ('ml', 'P_kbar', 'kbar'),
    ]):
        col_pred = f'{src}_pred_{tgt}'
        col_obs = f'obs_{tgt}'
        if col_pred not in pu.columns or col_obs not in pu.columns:
            ax.set_visible(False)
            continue
        y = pu[col_obs].values
        p = pu[col_pred].values
        mask = np.isfinite(y) & np.isfinite(p)
        color = PUTIRKA_COLOR if src == 'putirka' else ML_COLOR
        ax.scatter(y[mask], p[mask], s=14, alpha=0.6, color=color,
                   edgecolor='k', lw=0.3)
        one_to_one(ax)
        rmse = float(np.sqrt(mean_squared_error(y[mask], p[mask])))
        r2 = float(r2_score(y[mask], p[mask]))
        stats_box(ax, rmse=rmse, r2=r2, n=int(mask.sum()), unit=unit)
        ax.set_xlabel(f'Observed {unit}')
        ax.set_ylabel(f'Predicted {unit}')
        ax.set_title(f'{src.upper()} / {tgt}')
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_figure(fig, FIGURES / 'fig10_putirka_vs_ml',
                caption='Figure 10. Putirka (Thermobar) vs ML predictions on the opx-liq test set.')
    plt.close(fig)
else:
    print(f'skipping Fig 10: {putirka_path.name} not found')

In [ ]:
# Fig 11: bias-correction residuals (Phase 7R 4-track)
preds = pd.read_csv(RESULTS / 'nb07_test_predictions.csv')

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
panels = [
    ('T_C',   'C',    'T_true', 'T_rf_raw',     None,         None,         'Raw RF'),
    ('T_C',   'C',    'T_true', 'T_rf_pwcorr',  None,         None,         'Piecewise (OOF)'),
    ('T_C',   'C',    'T_true', 'T_qrf_median', 'T_qrf_lo',   'T_qrf_hi',   'QRF median + 68% band'),
    ('P_kbar','kbar', 'P_true', 'P_rf_raw',     None,         None,         'Raw RF'),
    ('P_kbar','kbar', 'P_true', 'P_rf_pwcorr',  None,         None,         'Piecewise (OOF)'),
    ('P_kbar','kbar', 'P_true', 'P_qrf_median', 'P_qrf_lo',   'P_qrf_hi',   'QRF median + 68% band'),
]
for ax, (tgt, unit, tcol, pcol, lo_col, hi_col, title) in zip(axes.flat, panels):
    y = preds[tcol].values
    p = preds[pcol].values
    res = p - y
    ax.scatter(y, res, s=16, alpha=0.6, c=TOL_BRIGHT['blue'], edgecolor='k', lw=0.3)
    if lo_col is not None and hi_col is not None:
        order = np.argsort(y)
        lo = preds[lo_col].values
        hi = preds[hi_col].values
        ax.fill_between(y[order], (lo - y)[order], (hi - y)[order],
                        alpha=0.2, color='gray', label='16-84%')
        ax.legend(loc='upper right', fontsize=8)
    ax.axhline(0, color='k', lw=0.7)
    rmse = float(np.sqrt(mean_squared_error(y, p)))
    ax.set_xlabel(f'True {unit}')
    ax.set_ylabel(f'Residual ({unit})')
    ax.set_title(f'{title}\nRMSE = {rmse:.2f}')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIGURES / 'fig11_bias_correction_residuals',
            caption='Figure 11. Residuals on the opx-liq test set for raw RF, piecewise-corrected RF, and QRF median with 16-84% band. Top row T, bottom row P.')
plt.close(fig)

In [ ]:
# Fig 12: ArcPL natural samples on a Hasterok & Chapman (2011) geotherm
ood_scores = pd.read_csv(RESULTS / 'nb10_arcpl_ood_scores.csv')

fig, ax = plt.subplots(figsize=(6, 6))
inlier = ood_scores['iso_inlier'] == 1
ax.scatter(ood_scores.loc[inlier, 'T_pred'], ood_scores.loc[inlier, 'P_pred'],
           s=20, alpha=0.7, c=TOL_BRIGHT['blue'], edgecolor='k', lw=0.3,
           label=f'in-dist (n={int(inlier.sum())})')
ax.scatter(ood_scores.loc[~inlier, 'T_pred'], ood_scores.loc[~inlier, 'P_pred'],
           s=20, alpha=0.7, c=TOL_BRIGHT['red'], edgecolor='k', lw=0.3, marker='x',
           label=f'OOD (n={int((~inlier).sum())})')

# Hasterok & Chapman (2011) reference geotherms at three surface heat flows
z_km = np.linspace(0, 250, 100)
for q_s, ls, col in [(40, '-', TOL_BRIGHT['gray']),
                     (60, '--', TOL_BRIGHT['gray']),
                     (80, ':',  TOL_BRIGHT['gray'])]:
    T_ref, P_ref = hasterok_chapman_geotherm(q_s, z_km)
    ax.plot(T_ref, P_ref, linestyle=ls, lw=1.0, color=col, alpha=0.8,
            label=f'H&C2011 q_s={q_s} mW/m2')

ax.set_xlabel('Predicted T (C)')
ax.set_ylabel('Predicted P (kbar)')
ax.invert_yaxis()
ax.set_title('ArcPL natural samples on H&C geotherm')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)
save_figure(fig, FIGURES / 'fig12_natural_samples_geotherm',
            caption='Figure 12. ArcPL natural samples projected into predicted '
                    'P-T space on Hasterok & Chapman (2011) layered conductive '
                    'geotherms for cratonic (40), average (60), and hot (80) '
                    'mW/m^2 surface heat flow. Points colored by IsolationForest '
                    'in-distribution flag.')
plt.close(fig)


# Fig 13: OOD score vs residual on ArcPL (where obs is available)
df_arcpl_path = RESULTS / 'nb04b_arcpl_predictions.csv'
if df_arcpl_path.exists():
    arc = pd.read_csv(df_arcpl_path)
else:
    raise FileNotFoundError(f"Missing {df_arcpl_path}. Run nb04b first.")
arc = arc.reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, tgt, unit in zip(axes, ['T_C', 'P_kbar'], ['C', 'kbar']):
    if tgt not in arc.columns:
        ax.set_visible(False)
        continue
    score = ood_scores['iso_score'].values
    pred = ood_scores[f'{tgt.split("_")[0]}_pred'].values
    obs = arc[tgt].values
    resid = pred - obs
    mask = np.isfinite(resid)
    ax.scatter(score[mask], resid[mask], s=16, alpha=0.6,
               c=TOL_BRIGHT['green'], edgecolor='k', lw=0.3)
    ax.axhline(0, color='k', lw=0.7)
    ax.set_xlabel('IsolationForest score (higher = in-dist)')
    ax.set_ylabel(f'Residual ({unit})')
    ax.set_title(f'{tgt} residual vs OOD score')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIGURES / 'fig13_ood_score_vs_residual',
            caption='Figure 13. IsolationForest OOD score against prediction residual for the ArcPL natural samples.')
plt.close(fig)


# Fig 14: MC analytical sigma vs tree-ensemble IQR
mc = pd.read_csv(RESULTS / 'nb10_mc_per_sample.csv')
# Need per-sample IQR from the canonical RF
df_test_liq = df_liq.loc[np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')].copy()
# Per-target feature sets (forest T=alr, forest P=raw for opx_liq).
from src.data import canonical_model_spec
_spec_T14 = canonical_model_spec('T_C',    'opx_liq', 'forest', RESULTS)
_spec_P14 = canonical_model_spec('P_kbar', 'opx_liq', 'forest', RESULTS)
X_liq_t_T, _ = build_feature_matrix(df_test_liq, _spec_T14['feature_set'], use_liq=True)
X_liq_t_P, _ = build_feature_matrix(df_test_liq, _spec_P14['feature_set'], use_liq=True)
m_T = joblib.load(MODELS / canonical_model_filename('T_C', 'opx_liq', 'forest', RESULTS))
m_P = joblib.load(MODELS / canonical_model_filename('P_kbar', 'opx_liq', 'forest', RESULTS))

iqr_T = predict_iqr(m_T, X_liq_t_T)[2] - predict_iqr(m_T, X_liq_t_T)[0]
iqr_P = predict_iqr(m_P, X_liq_t_P)[2] - predict_iqr(m_P, X_liq_t_P)[0]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].scatter(mc['T_sigma'], iqr_T, s=16, alpha=0.6, c=TOL_BRIGHT['purple'],
                edgecolor='k', lw=0.3)
axes[0].set_xlabel('MC analytical sigma T (C)')
axes[0].set_ylabel('Tree IQR T (C)')
axes[0].set_title('T uncertainty: analytical vs tree-ensemble')
axes[0].grid(True, alpha=0.3)
axes[1].scatter(mc['P_sigma'], iqr_P, s=16, alpha=0.6, c=TOL_BRIGHT['purple'],
                edgecolor='k', lw=0.3)
axes[1].set_xlabel('MC analytical sigma P (kbar)')
axes[1].set_ylabel('Tree IQR P (kbar)')
axes[1].set_title('P uncertainty: analytical vs tree-ensemble')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIGURES / 'fig14_mc_vs_iqr_uncertainty',
            caption='Figure 14. Per-sample Monte-Carlo analytical uncertainty against tree-ensemble IQR for T and P.')
plt.close(fig)
print('=== NBF complete ===')
